In [0]:
%sql
MERGE INTO dbw_maritime_2026.maritime_showcase.silver_ship_current_state AS target
USING (
  SELECT mmsi, ship_name, latitude, longitude, current_port_vicinity, event_time_utc
  FROM (
    SELECT *, ROW_NUMBER() OVER(PARTITION BY mmsi ORDER BY event_time_utc DESC) as rn
    FROM dbw_maritime_2026.maritime_showcase.silver_ais_enriched
  ) WHERE rn = 1
) AS source
ON target.mmsi = source.mmsi
WHEN MATCHED AND source.event_time_utc > target.last_event_time_utc THEN
  UPDATE SET target.latitude = source.latitude, target.longitude = source.longitude, target.current_port_vicinity = source.current_port_vicinity, target.last_event_time_utc = source.event_time_utc, target.update_timestamp = current_timestamp()
WHEN NOT MATCHED THEN
  INSERT (mmsi, ship_name, latitude, longitude, current_port_vicinity, last_event_time_utc, update_timestamp)
  VALUES (source.mmsi, source.ship_name, source.latitude, source.longitude, source.current_port_vicinity, source.event_time_utc, current_timestamp());